# SCXML State Machine Co-Simulation

This example shows how to use Python SXCML library (https://github.com/Open-MBEE/scxml4py)
to simulate a state machine defined in SCXML.


Import all dependencies needed for the example.


In [ ]:
import logging
import time
import scxml4py.helper
from scxmlApp.application import Application
from scxml4py.action import Action
from scxml4py.activity import ThreadedActivity
from scxml4py.event import Event

## Define State Machine

In [2]:
simpleSM_string = '''<?xml version="1.0" ?>
<scxml xmlns="http://www.w3.org/2005/07/scxml" version="1.0" datamodel="ecmascript" initial="OFF">
  <state id="OFF">
    <transition event="GoOnline" target="ONLINE"/>
  </state>
  <state id="ONLINE">
    <invoke id="ActivityOnline"/>
    <transition event="GoOffline" target="OFF"/>
  </state>
</scxml>
'''


# Set up SCXML simulation environment

## Declare SCXML Listeners and do actions

In [3]:

# Installed as status listener, engine looks for "ActionStatusListener" action
class ActionStatusListener(Action):
    # implemented by the developer, stub can be generated
    def __init__(self, theData):
        Action.__init__(self, "ActionStatusListener", None, theData)
    
    def execute(self, status):
        logging.getLogger("scxml4py").info(">>>>ActionStatusListener::nb Status: <" + scxml4py.helper.formatStatus(status) + ">")

class ActionEventListener(Action):
    # implemented by the developer, stub can be generated
    def __init__(self, theData):
        Action.__init__(self, "ActionEventListener", None, theData)
    
    def execute(self, event):
        logging.getLogger("scxml4py").info(">>>>ActionEventListener::nb Event: <" + ">")
        print (event.getStatus)

# Do Action that gets invoked when entering state, name must must do action in v2 model
class ActivityOnline(ThreadedActivity):
    # implemented by the developer, stub can be generated
    def __init__(self, theEventQueue, theData):
        ThreadedActivity.__init__(self, "ActivityOnline", theEventQueue, theData)

    def run(self):
        counter = 0
        while self.isRunning() == True:
            logging.getLogger("scxml4py").info("Activity <" + self.getId() + f"> is running...{counter}")
            time.sleep(2)
            counter += 1
            if counter == 5:
                self.sendInternalEvent(Event("GoOffline"))
                self.setRunning(False)
                break
            
# Data Object, can be generated from attributes of owning part that exhibits behavior
class Data(object):
    # implemented by the developer
    # data shared between actions and activities
    def __init__(self):
        self.mSharedInfo = None
    
    def getSharedInfo(self):
        # requires mutex
        return self.mSharedInfo
    
    def setSharedInfo(self, sharedInfo):
        # requires mutex
        self.mSharedInfo = sharedInfo

## Run the Simulation

In [4]:
logging.basicConfig(format='%(asctime)s - %(levelname)s - %(threadName)s - %(module)s - %(funcName)s - %(message)s', level=logging.DEBUG)
simple_sm_data = Data()
simple_sm = Application(simpleSM_string, actions = [ActionStatusListener,ActionEventListener], activities = [ActivityOnline], data = simple_sm_data)
simple_sm.start()
simple_sm.send_signal("GoOnline", True)
status = simple_sm.get_current_status()
print (">>>>>>" + status)
time.sleep(10)
simple_sm.send_signal("GoOffline", True)
status = simple_sm.get_current_status()
print (">>>>>>" + status)
simple_sm.terminate()

2026-03-12 21:16:06,796 - INFO - MainThread - application - __init__ - _Application::Loading SCXML model
2026-03-12 21:16:06,797 - DEBUG - MainThread - reader - parseScxml - Version: 1.0
2026-03-12 21:16:06,799 - DEBUG - MainThread - reader - parseScxml - Optional name attribute not available.
2026-03-12 21:16:06,801 - DEBUG - MainThread - reader - parseStates - Found new state: OFF
2026-03-12 21:16:06,802 - DEBUG - MainThread - reader - parseStates - Found new state: ONLINE
2026-03-12 21:16:06,804 - DEBUG - MainThread - reader - parseScxmlInitialTransition - Found initial transition to state: OFF
2026-03-12 21:16:06,805 - DEBUG - MainThread - reader - parseTransitions - Found new transition to state: ONLINE
2026-03-12 21:16:06,806 - DEBUG - MainThread - reader - parseTransitions - Found new transition to state: OFF
2026-03-12 21:16:06,807 - DEBUG - MainThread - reader - parseInvokes - Found activity ActivityOnline for state ONLINE
2026-03-12 21:16:06,808 - DEBUG - MainThread - applica

>>>>>>ONLINE
<bound method Event.getStatus of <scxml4py.event.Event object at 0x7d0ff4dbac30>>


2026-03-12 21:16:08,829 - INFO - Thread-6 (run) - 4202266828 - run - Activity <ActivityOnline> is running...1
2026-03-12 21:16:10,830 - INFO - Thread-6 (run) - 4202266828 - run - Activity <ActivityOnline> is running...2
2026-03-12 21:16:12,832 - INFO - Thread-6 (run) - 4202266828 - run - Activity <ActivityOnline> is running...3
2026-03-12 21:16:14,833 - INFO - Thread-6 (run) - 4202266828 - run - Activity <ActivityOnline> is running...4
2026-03-12 21:16:16,826 - DEBUG - Thread-5 - application - run - _Application::Application received event = <GoOffline>
2026-03-12 21:16:16,828 - DEBUG - Thread-5 - executor - processEvent - Adding event <GoOffline> to the external queue
2026-03-12 21:16:16,828 - DEBUG - Thread-5 - executor - processEvents - Processing external event <GoOffline>
2026-03-12 21:16:16,829 - DEBUG - Thread-5 - executor - selectTransitions - Selected transitions on event <GoOffline>:
FromState <ONLINE> ToState <OFF>  Guards <> Event <GoOffline> Actions <>

2026-03-12 21:16:16

>>>>>>ONLINE
<bound method Event.getStatus of <scxml4py.event.Event object at 0x7d0ff4dbaf90>>
<bound method Event.getStatus of <scxml4py.event.Event object at 0x7d0ff4dba780>>
